<a href="https://colab.research.google.com/github/algulawani/RM-Research-Final-Task/blob/main/get_landmarks_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
This script computes the 24 interest points that the OADN model
uses for its landmark-guided attention branch.

It loads the previously saved dictionary of 68 landmarks per image,
then for each image:
1. Selects 16 landmarks at specific indices covering the eyebrows,
   eyes, nose, and mouth
2. Computes 8 additional midpoints:
   - 4 between the eyes and eyebrows
   - 4 on the cheeks (between jaw and eye/mouth landmarks)
3. For each derived midpoint, the confidence score is set to the
   minimum of the two source landmarks' scores, as described in
   Section 3.1 of the OADN paper

The 16 selected landmark indices are:
  Eyebrows: 17, 19, 21, 22, 24, 26
  Eyes: 36, 39, 42, 45
  Nose: 27, 30, 33
  Mouth: 48, 51, 54

The 8 derived midpoints are computed from these pairs:
  Eye-eyebrow region:
    (17, 36), (19, 37), (24, 44), (26, 45)
  Cheeks:
    (1, 36), (3, 48), (15, 45), (13, 54)

Note: landmarks 37 and 44 (upper eyelids) and 1, 3, 13, 15 (jaw)
are not in the selected set but are used from the original 68
landmarks to compute the midpoints.

The result is a new dictionary saved as a .pth file with:
- Keys: image paths like "1/train_05178_aligned.jpg"
- Values: dictionaries with:
  - "landmarks": numpy array of shape (24, 2)
  - "scores": numpy array of shape (24,)
"""

import os
import numpy as np
import torch

input_path = "/content/drive/MyDrive/OADN/Preprocessed Testing Data/test_landmarks_confidencescores.pth"
output_dir = "/content/drive/MyDrive/OADN/Preprocessed Testing Data"
output_path = os.path.join(output_dir, "test_24_interest_points.pth")

landmarks_68 = torch.load(input_path, weights_only=False)

selected_indices = [17, 19, 21, 22, 24, 26, 36, 39, 42, 45, 27, 30, 33, 48, 51, 54]

midpoint_pairs = [
    (17, 36),  # left eyebrow start — left eye outer
    (19, 37),  # left eyebrow middle — left eye upper
    (24, 44),  # right eyebrow middle — right eye upper
    (26, 45),  # right eyebrow end — right eye outer
    (1, 36),   # jaw left — left eye outer (left cheek upper)
    (3, 48),   # jaw left — mouth left corner (left cheek lower)
    (15, 45),  # jaw right — right eye outer (right cheek upper)
    (13, 54),  # jaw right — mouth right corner (right cheek lower)
]

interest_points_dict = {}

for image_key in landmarks_68:
    all_landmarks = landmarks_68[image_key]["landmarks"]
    all_scores = landmarks_68[image_key]["scores"]

    # Get the 16 selected landmarks and their scores
    selected_landmarks = all_landmarks[selected_indices]
    selected_scores = all_scores[selected_indices]

    # Compute the 8 midpoints and their scores
    derived_landmarks = []
    derived_scores = []
    for idx_a, idx_b in midpoint_pairs:
        midpoint = (all_landmarks[idx_a] + all_landmarks[idx_b]) / 2
        score = min(all_scores[idx_a], all_scores[idx_b])
        derived_landmarks.append(midpoint)
        derived_scores.append(score)

    derived_landmarks = np.array(derived_landmarks)
    derived_scores = np.array(derived_scores)

    # Combine selected and derived into 24 interest points
    final_landmarks = np.concatenate([selected_landmarks, derived_landmarks], axis=0)
    final_scores = np.concatenate([selected_scores, derived_scores], axis=0)

    interest_points_dict[image_key] = {
        "landmarks": final_landmarks,
        "scores": final_scores
    }

torch.save(interest_points_dict, output_path)

print(f"Processed {len(interest_points_dict)} images")
print(f"Each image has {interest_points_dict[list(interest_points_dict.keys())[0]]['landmarks'].shape[0]} interest points")
print(f"Saved to: {output_path}")

Processed 3034 images
Each image has 24 interest points
Saved to: /content/drive/MyDrive/OADN/Preprocessed Testing Data/test_24_interest_points.pth
